In [1]:
import sys, json; sys.path.append("..")
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from src.train import make_logreg, make_lgbm
from src.evaluate import report

train = pd.read_csv("../data/processed/train.csv", index_col=0)
test = pd.read_csv("../data/processed/test.csv", index_col=0)
y_tr = train.pop("SeriousDlqin2yrs"); X_tr = train
y_te = test.pop("SeriousDlqin2yrs"); X_te = test

COST_FP, COST_FN = 1000, 6000
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best = json.load(open("../reports/best_lgbm_params.json"))
models = {"logreg": make_logreg(), "lightgbm": make_lgbm(**best)}

def cost(y_true, flagged):
    fp = (flagged & (y_true == 0)).sum()
    fn = (~flagged & (y_true == 1)).sum()
    return fp * COST_FP + fn * COST_FN

rows = {}
for name, model in models.items():
    # 1) choose the cutoff using TRAINING out-of-fold predictions only
    oof = cross_val_predict(model, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    grid = np.arange(0.05, 0.96, 0.05)
    t = grid[np.argmin([cost(y_tr, oof >= g) for g in grid])]
    # 2) fit on all training data, 3) score the sealed test set once
    model.fit(X_tr, y_tr)
    p = model.predict_proba(X_te)[:, 1]
    r = report(y_te, (p >= t).astype(int), p)
    r["threshold"] = round(t, 2)
    r["cost_millions"] = cost(y_te, p >= t) / 1e6
    rows[name] = r

pd.DataFrame(rows).T.round(3)

,accuracy,precision,recall,roc_auc,pr_auc,TN,FP,FN,TP,threshold,cost_millions
logreg,0.896,0.335,0.563,0.831,0.368,25755.0,2240.0,876.0,1129.0,0.6,7.496
lightgbm,0.897,0.344,0.599,0.870,0.401,25706.0,2289.0,804.0,1201.0,0.7,7.113
